In [1]:
import numpy as np
import pandas as pd

meta_data = pd.read_csv(r"C:\Users\USER\Desktop\資訊所實習\計畫\資料探勘\Ravindra2021.raw_count.stdprep.h5ad的細胞標籤\將mock拿掉重跑\GCN_metadata.csv")

In [2]:
print(meta_data.head())

                         Unnamed: 0           ctype Condition  \
0  AAACCCACAACCGCTG-1-1dpi_CoV2_HCR  Ciliated cells      1dpi   
1  AAACCCACACACAGAG-1-1dpi_CoV2_HCR     Basal cells      1dpi   
2  AAACCCACATCTAACG-1-1dpi_CoV2_HCR  Ciliated cells      1dpi   
3  AAACCCAGTCCCGGTA-1-1dpi_CoV2_HCR      Tuft cells      1dpi   
4  AAACCCAGTTAGCGGA-1-1dpi_CoV2_HCR      Club cells      1dpi   

   Viral_transcript  label                label_source  
0                 0      0  high_confidence_uninfected  
1                 0      0  high_confidence_uninfected  
2                 2     -1                     unknown  
3                 0      0  high_confidence_uninfected  
4                 0      0  high_confidence_uninfected  


In [3]:
top_3000_gene = pd.read_csv(r"C:\Users\USER\Desktop\資訊所實習\計畫\資料探勘\Ravindra2021.raw_count.stdprep.h5ad的細胞標籤\將mock拿掉重跑\GCN_genes.csv")

In [4]:
print(top_3000_gene.head())

          0
0     ISG15
1    TTLL10
2  C1QTNF12
3    TAS1R3
4    CFAP74


In [5]:
import scanpy as sc

adata = sc.read_h5ad(r"C:\Users\USER\Desktop\資訊所實習\計畫\資料探勘\Ravindra2021.raw_count.stdprep.h5ad")

In [7]:
unknown_prediction = pd.read_csv(r"C:\Users\USER\Desktop\資訊所實習\計畫\資料探勘\Ravindra2021.raw_count.stdprep.h5ad的細胞標籤\將mock拿掉重跑\其他機器學習模型\Logistic_unknown_prediction.csv")

In [8]:
print(unknown_prediction.head())

                         Unnamed: 0           ctype Condition  \
0  AAACCCACATCTAACG-1-1dpi_CoV2_HCR  Ciliated cells      1dpi   
1  AAACCCATCGACGATT-1-1dpi_CoV2_HCR      Club cells      1dpi   
2  AAACGAACAGAGTCAG-1-1dpi_CoV2_HCR  Ciliated cells      1dpi   
3  AAACGAACAGCCCACA-1-1dpi_CoV2_HCR      Club cells      1dpi   
4  AAACGAAGTTCGATTG-1-1dpi_CoV2_HCR      Club cells      1dpi   

   Viral_transcript  label label_source  predict_label  infect_prob  
0                 2     -1      unknown              0     0.112997  
1                 1     -1      unknown              0     0.000013  
2                 1     -1      unknown              0     0.000087  
3                 1     -1      unknown              0     0.000032  
4                 0     -1      unknown              0     0.000047  


In [9]:
ct = pd.crosstab(
    unknown_prediction["ctype"],
    unknown_prediction["predict_label"]
)

print(ct)
from scipy.stats import chi2_contingency

chi2, p, dof, expected = chi2_contingency(ct)

print("Chi-square:", chi2)
print("df:", dof)
print("p-value:", p)


predict_label            0      1
ctype                            
BC/Club                875   1161
Basal cells           4097  13863
Ciliated cells        6403   5551
Club cells            6429   3429
Goblet cells            43     21
Ionocytes              321    133
Neuroendocrine cells   173     54
Tuft cells             145     71
Chi-square: 5818.569228372369
df: 7
p-value: 0.0


In [10]:
infection_rate = (
    unknown_prediction
    .groupby("ctype")["predict_label"]
    .mean()
    .sort_values(ascending=False)
)

print(infection_rate)

ctype
Basal cells             0.771882
BC/Club                 0.570236
Ciliated cells          0.464363
Club cells              0.347839
Tuft cells              0.328704
Goblet cells            0.328125
Ionocytes               0.292952
Neuroendocrine cells    0.237885
Name: predict_label, dtype: float64


In [11]:
# 比較GCN與Logistic針對unknown cell預測結果的不同

GCN_result = pd.read_csv(r"C:\Users\USER\Desktop\資訊所實習\計畫\資料探勘\Ravindra2021.raw_count.stdprep.h5ad的細胞標籤\將mock拿掉重跑\GCN_unknown_prediction.csv")

Log_result = unknown_prediction.copy()

In [13]:
print(GCN_result.head())

print(Log_result.head())

                         Unnamed: 0           ctype Condition  \
0  AAACCCACATCTAACG-1-1dpi_CoV2_HCR  Ciliated cells      1dpi   
1  AAACCCATCGACGATT-1-1dpi_CoV2_HCR      Club cells      1dpi   
2  AAACGAACAGAGTCAG-1-1dpi_CoV2_HCR  Ciliated cells      1dpi   
3  AAACGAACAGCCCACA-1-1dpi_CoV2_HCR      Club cells      1dpi   
4  AAACGAAGTTCGATTG-1-1dpi_CoV2_HCR      Club cells      1dpi   

   Viral_transcript  label label_source  predict_label  infect_prob  
0                 2     -1      unknown              0     0.000005  
1                 1     -1      unknown              0     0.000013  
2                 1     -1      unknown              0     0.000003  
3                 1     -1      unknown              0     0.004681  
4                 0     -1      unknown              0     0.000023  
                         Unnamed: 0           ctype Condition  \
0  AAACCCACATCTAACG-1-1dpi_CoV2_HCR  Ciliated cells      1dpi   
1  AAACCCATCGACGATT-1-1dpi_CoV2_HCR      Club cells      1d

In [16]:
print("Row 數量是否相同：",
      len(GCN_result) == len(Log_result))

print("Cell ID 是否完全相同且順序一致：",
      GCN_result["Unnamed: 0"].equals(
          Log_result["Unnamed: 0"]
      ))

print("ctype 是否完全一致：",
      GCN_result["ctype"].equals(
          Log_result["ctype"]
      ))

print("Condition 是否完全一致：",
      GCN_result["Condition"].equals(
          Log_result["Condition"]
      ))

print("Viral_transcript 是否完全一致：",
      GCN_result["Viral_transcript"].equals(
          Log_result["Viral_transcript"]
      ))

Row 數量是否相同： True
Cell ID 是否完全相同且順序一致： True
ctype 是否完全一致： True
Condition 是否完全一致： True
Viral_transcript 是否完全一致： True


In [17]:
# 兩個模型是否做出相同的感染/未感染判斷
same_prediction = (
    GCN_result["predict_label"] == Log_result["predict_label"]
)

print("預測相同的 cell 數量：", same_prediction.sum())
print("預測不同的 cell 數量：", (~same_prediction).sum())
print("預測一致率：", same_prediction.mean())

預測相同的 cell 數量： 34383
預測不同的 cell 數量： 8386
預測一致率： 0.8039234024644017


In [18]:
comparison = pd.crosstab(
    GCN_result["predict_label"],
    Log_result["predict_label"],
    rownames=["GCN"],
    colnames=["Logistic"]
)

print(comparison)

Logistic      0      1
GCN                   
0         16523   6423
1          1963  17860


In [19]:
disagreement = GCN_result[
    GCN_result["predict_label"] != Log_result["predict_label"]
].copy()

print("模型預測不同的 cell 數量：", len(disagreement))

print(
    disagreement[
        [
            "Unnamed: 0",
            "ctype",
            "Condition",
            "Viral_transcript",
            "predict_label",
            "infect_prob"
        ]
    ].head(20)
)

模型預測不同的 cell 數量： 8386
                           Unnamed: 0           ctype Condition  \
63   AACGGGAAGGCGATAC-1-1dpi_CoV2_HCR  Ciliated cells      1dpi   
67   AACGGGAGTCCCTCAT-1-1dpi_CoV2_HCR  Ciliated cells      1dpi   
148  AATTCCTAGGAAAGAC-1-1dpi_CoV2_HCR  Ciliated cells      1dpi   
209  ACCCTTGCATATGCGT-1-1dpi_CoV2_HCR  Ciliated cells      1dpi   
273  ACTGATGGTCAGATTC-1-1dpi_CoV2_HCR  Ciliated cells      1dpi   
334  AGACAGGAGAGCAGAA-1-1dpi_CoV2_HCR  Ciliated cells      1dpi   
398  AGCGATTGTTGTCCCT-1-1dpi_CoV2_HCR  Ciliated cells      1dpi   
492  AGTACCAAGGAGTCTG-1-1dpi_CoV2_HCR  Ciliated cells      1dpi   
497  AGTACTGGTGTATACC-1-1dpi_CoV2_HCR  Ciliated cells      1dpi   
523  AGTGATCGTCCAATCA-1-1dpi_CoV2_HCR     Basal cells      1dpi   
524  AGTGATCTCAGGACAG-1-1dpi_CoV2_HCR  Ciliated cells      1dpi   
574  ATCATTCGTCTTCATT-1-1dpi_CoV2_HCR  Ciliated cells      1dpi   
601  ATCGTAGGTATATGGA-1-1dpi_CoV2_HCR      Club cells      1dpi   
691  ATTCTACTCCTACCGT-1-1dpi_CoV2_HCR  C

In [20]:
comparison_df = GCN_result[
    [
        "Unnamed: 0",
        "ctype",
        "Condition",
        "Viral_transcript"
    ]
].copy()

comparison_df["GCN_label"] = GCN_result["predict_label"]
comparison_df["GCN_prob"] = GCN_result["infect_prob"]

comparison_df["Logistic_label"] = Log_result["predict_label"]
comparison_df["Logistic_prob"] = Log_result["infect_prob"]

print(comparison_df.head())

                         Unnamed: 0           ctype Condition  \
0  AAACCCACATCTAACG-1-1dpi_CoV2_HCR  Ciliated cells      1dpi   
1  AAACCCATCGACGATT-1-1dpi_CoV2_HCR      Club cells      1dpi   
2  AAACGAACAGAGTCAG-1-1dpi_CoV2_HCR  Ciliated cells      1dpi   
3  AAACGAACAGCCCACA-1-1dpi_CoV2_HCR      Club cells      1dpi   
4  AAACGAAGTTCGATTG-1-1dpi_CoV2_HCR      Club cells      1dpi   

   Viral_transcript  GCN_label  GCN_prob  Logistic_label  Logistic_prob  
0                 2          0  0.000005               0       0.112997  
1                 1          0  0.000013               0       0.000013  
2                 1          0  0.000003               0       0.000087  
3                 1          0  0.004681               0       0.000032  
4                 0          0  0.000023               0       0.000047  


In [21]:
print("GCN 預測感染數：",
      (comparison_df["GCN_label"] == 1).sum())

print("Logistic 預測感染數：",
      (comparison_df["Logistic_label"] == 1).sum())

print()

print("GCN 預測感染率：",
      (comparison_df["GCN_label"] == 1).mean())

print("Logistic 預測感染率：",
      (comparison_df["Logistic_label"] == 1).mean())

GCN 預測感染數： 19823
Logistic 預測感染數： 24283

GCN 預測感染率： 0.46348991091678554
Logistic 預測感染率： 0.5677710491243658


In [22]:
celltype_comparison = (
    comparison_df
    .groupby("ctype")
    .agg(
        n_cells=("Unnamed: 0", "size"),
        GCN_infected=("GCN_label", "sum"),
        Logistic_infected=("Logistic_label", "sum")
    )
)

celltype_comparison["GCN_infection_rate"] = (
    celltype_comparison["GCN_infected"]
    / celltype_comparison["n_cells"]
)

celltype_comparison["Logistic_infection_rate"] = (
    celltype_comparison["Logistic_infected"]
    / celltype_comparison["n_cells"]
)

print(celltype_comparison)

                      n_cells  GCN_infected  Logistic_infected  \
ctype                                                            
BC/Club                  2036          1160               1161   
Basal cells             17960         10137              13863   
Ciliated cells          11954          5764               5551   
Club cells               9858          2707               3429   
Goblet cells               64            17                 21   
Ionocytes                 454            27                133   
Neuroendocrine cells      227             2                 54   
Tuft cells                216             9                 71   

                      GCN_infection_rate  Logistic_infection_rate  
ctype                                                              
BC/Club                         0.569745                 0.570236  
Basal cells                     0.564421                 0.771882  
Ciliated cells                  0.482182                 0.464363  